# Local Invoice OCR on Google Colab
Run every cell in order. Public repositories need no token; for a private repository add a Colab secret named `GITHUB_TOKEN` with read access and enable notebook access. A GPU runtime is recommended: **Runtime > Change runtime type > T4 GPU**. The first run downloads PaddleOCR models and, when enabled, the Ollama model.

In [ ]:
#@title 1. Download the project
import io, os, pathlib, shutil, urllib.error, urllib.request, zipfile
from google.colab import userdata
PROJECT_DIR = pathlib.Path('/content/OCR')
ARCHIVE_URL = 'https://api.github.com/repos/ubaid-148/OCR/zipball/main'
headers = {'Accept': 'application/vnd.github+json', 'User-Agent': 'OCR-Colab-Setup'}
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None
if github_token:
    headers['Authorization'] = f'Bearer {github_token}'
request = urllib.request.Request(ARCHIVE_URL, headers=headers)
try:
    with urllib.request.urlopen(request, timeout=120) as response:
        archive_bytes = response.read()
except urllib.error.HTTPError as error:
    raise RuntimeError(f'GitHub download failed ({error.code}). Make the repo public or grant GITHUB_TOKEN read access.') from error
extract_root = pathlib.Path('/content/ocr-download')
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(PROJECT_DIR, ignore_errors=True)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(extract_root)
extracted = next(path for path in extract_root.iterdir() if path.is_dir())
shutil.move(str(extracted), str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print('Private project ready at', PROJECT_DIR)

In [ ]:
#@title 2. Install OCR and Python dependencies (first run takes several minutes)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-eng tesseract-ocr-ara tesseract-ocr-urd ghostscript unpaper pngquant
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
#@title 3. Start local AI (Ollama)
USE_LOCAL_AI = True #@param {type:"boolean"}
OLLAMA_MODEL = 'qwen2.5:3b' #@param {type:"string"}
import os, shutil, subprocess, time, urllib.request
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
if USE_LOCAL_AI:
    if not shutil.which('ollama'):
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    ollama_log = open('/tmp/ollama.log', 'w')
    ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('Ollama ready:', OLLAMA_MODEL)
else:
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
    print('Local AI disabled; deterministic spatial fallback will be used.')

In [ ]:
#@title 4. Start OCR web application
import os, subprocess, sys, time, urllib.request
os.chdir('/content/OCR')
if 'ocr_process' in globals() and ocr_process.poll() is None:
    ocr_process.terminate()
ocr_log = open('/tmp/ocr-web.log', 'w')
ocr_process = subprocess.Popen([sys.executable, 'ocr_web.py'], stdout=ocr_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(120):
    try:
        response = urllib.request.urlopen('http://127.0.0.1:8765/', timeout=2)
        if response.status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ocr-web.log').read())
    raise RuntimeError('OCR web application did not start')
print('OCR application is ready.')

In [ ]:
#@title 5. Open the application
from google.colab import output
output.serve_kernel_port_as_iframe(8765, height='700')

Upload a PDF in the embedded application. Colab storage and processes are temporary; rerun the notebook after a runtime reset. If startup fails, inspect `!/tmp/ocr-web.log` or `!/tmp/ollama.log`.